# Cuadernillo de trabajo: Meningitis Dataset (con valores faltantes)

Este cuaderno prepara el archivo `mening missing 12.csv` para tareas de **regresion**, **clasificacion** y **redes neuronales**.

Objetivos:
- Entender el contexto general del problema.
- Documentar que significa cada columna.
- Analizar calidad de datos (tipos, faltantes, distribuciones).
- Transformar el dataset a formato numerico de forma explicada y reproducible.
- Dejar matrices listas para modelado.


## 1) Contexto general

La meningitis es una inflamacion de las membranas que rodean el cerebro y la medula espinal. En un entorno clinico, los datos de laboratorio y variables demograficas permiten:

- Estimar gravedad y riesgo del paciente.
- Distinguir entre posibles etiologias (por ejemplo, bacteriana o viral).
- Predecir desenlaces clinicos.

Este dataset combina variables clinicas y de laboratorio con etiquetas de diagnostico/riesgo. Como contiene valores faltantes y variables categoricas, requiere preprocesamiento antes de entrenar modelos de machine learning.


## 2) Diccionario de columnas

| Columna | Tipo esperado | Significado |
|---|---|---|
| Patient_ID | Identificador | ID unico del paciente (no representa condicion clinica por si solo). |
| Age | Numerica | Edad del paciente. |
| Gender | Categorica | Sexo del paciente (`Male`, `Female`). |
| WBC_Count | Numerica | Recuento de leucocitos en muestra relacionada al cuadro de meningitis. |
| Protein_Level | Numerica | Nivel de proteinas en la muestra clinica. |
| Glucose_Level | Numerica | Nivel de glucosa en la muestra clinica. |
| Pathogen_Present | Categorica binaria | Presencia de patogeno (`Yes`/`No`). |
| Diagnosis | Categorica | Tipo de diagnostico (`Bacterial`, `Viral`, `Unknown`). |
| Outcome | Categorica binaria | Desenlace clinico (`Recovered`, `Deceased`). |
| Hemoglobin | Numerica | Nivel de hemoglobina. |
| WBC_Blood_Count | Numerica | Recuento de leucocitos en sangre periferica. |
| Platelets | Numerica | Recuento plaquetario. |
| CRP_Level | Numerica | Proteina C reactiva (marcador inflamatorio). |
| Risk_Level | Categorica ordinal | Nivel de riesgo (`Low Risk`, `Moderate Risk`, `High Risk`). |


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, LabelEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [ ]:
# Carga de datos
file_path = 'mening missing 12.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
display(df.head())

## 3) Diagnostico inicial del dataset

In [ ]:
print('Tipos de datos:')
display(df.dtypes.to_frame('dtype'))

print('Valores faltantes por columna:')
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing.sort_values('missing_count', ascending=False))

In [ ]:
# Revision de categorias
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for c in cat_cols:
    print(f'\n{c}:', sorted(df[c].dropna().unique().tolist()))

## 4) Estrategia de transformacion a numerico (explicada)

Se aplicaran las siguientes decisiones:

1. **Eliminar `Patient_ID` como predictor**
   - Motivo: es un identificador tecnico, no una variable clinica causal. Puede introducir ruido o sobreajuste.

2. **Imputacion de faltantes**
   - Columnas numericas: imputacion por **mediana** (robusta ante valores extremos).
   - Columnas categoricas: imputacion por **moda** (valor mas frecuente).

3. **Codificacion de texto a numerico**
   - Variables categoricas nominales (`Gender`, `Pathogen_Present`, `Diagnosis`, `Outcome`) se transforman con **One-Hot Encoding**.
   - Variable ordinal `Risk_Level` se transforma con orden clinico:
     - `Low Risk` -> 0
     - `Moderate Risk` -> 1
     - `High Risk` -> 2

4. **Escalado para modelos sensibles a magnitud**
   - Variables numericas se escalan con **StandardScaler** (media 0, desviacion estandar 1), especialmente util para redes neuronales y algunos clasificadores/regresores.

5. **Preparacion segun objetivo de modelado**
   - **Clasificacion**: usar como ejemplo objetivo `Outcome` o `Diagnosis`.
   - **Regresion**: usar un objetivo numerico (por ejemplo `CRP_Level`) si se desea predecir valor continuo.
   - **Redes neuronales**: utilizar el mismo `X` numerico resultante del preprocesamiento.


In [ ]:
# Copia de trabajo
data = df.copy()

# 1) Eliminar ID tecnico
if 'Patient_ID' in data.columns:
    data = data.drop(columns=['Patient_ID'])

# 2) Codificacion ordinal explicita para Risk_Level
risk_map = {
    'Low Risk': 0,
    'Moderate Risk': 1,
    'High Risk': 2
}
data['Risk_Level_Ordinal'] = data['Risk_Level'].map(risk_map)

# Mantenemos la columna original para inspeccion y luego decidimos usar una u otra segun tarea
print('Columnas tras ajustes iniciales:')
print(data.columns.tolist())

display(data.head())

In [ ]:
# Definir objetivo de ejemplo para CLASIFICACION
# Puedes cambiar target_col por 'Diagnosis', 'Outcome' o incluso 'Risk_Level' segun tu ejercicio.
target_col = 'Outcome'

# Si el objetivo es categorico, no debe formar parte de X
y_raw = data[target_col]
X = data.drop(columns=[target_col])

# Evitamos duplicidad: para modelado usamos la version ordinal de riesgo y removemos la textual
if 'Risk_Level' in X.columns and 'Risk_Level_Ordinal' in X.columns:
    X = X.drop(columns=['Risk_Level'])

# Identificar tipos de columnas en X
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

print('Target:', target_col)
print('Numericas en X:', num_cols)
print('Categoricas en X:', cat_cols)

In [ ]:
# Codificar y (clasificacion) a numerico si hace falta
if y_raw.dtype in ['object', 'string']:
    y_encoder = LabelEncoder()
    y = y_encoder.fit_transform(y_raw.astype(str))
    print('Clases del target codificadas:')
    for i, cls in enumerate(y_encoder.classes_):
        print(f'  {cls} -> {i}')
else:
    y = y_raw.values
    y_encoder = None

# Preprocesador para X
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
    ],
    remainder='drop'
)

X_processed = preprocessor.fit_transform(X)

print('Forma de X original:', X.shape)
print('Forma de X procesado:', X_processed.shape)
print('Tipo de X procesado:', type(X_processed))

In [ ]:
# Split listo para ejercicios de ML
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(np.unique(y)) < 20 else None
)

print('Train shapes:', X_train.shape, y_train.shape)
print('Test shapes:', X_test.shape, y_test.shape)

## 5) Que columnas se transformaron y por que

### A) Reemplazos por imputacion
- **Numericas** (`Age`, `WBC_Count`, `Protein_Level`, `Glucose_Level`, `Hemoglobin`, `WBC_Blood_Count`, `Platelets`, `CRP_Level`, `Risk_Level_Ordinal`):
  - Se reemplazan faltantes por la **mediana** de cada columna.
  - Razon: conserva tendencia central sin depender fuertemente de outliers.

- **Categoricas** (`Gender`, `Pathogen_Present`, `Diagnosis`, y cualquier otra textual en `X`):
  - Se reemplazan faltantes por la **moda** de cada columna.
  - Razon: mantiene una categoria valida y evita perder filas.

### B) Conversiones de texto a numero
- `Risk_Level` (texto) se transforma a `Risk_Level_Ordinal` con mapeo explicito:
  - `Low Risk` = 0, `Moderate Risk` = 1, `High Risk` = 2.
  - Razon: existe orden natural de severidad.

- Variables categoricas nominales se convierten por One-Hot Encoding:
  - Ejemplo: `Gender_Female`, `Gender_Male`; `Pathogen_Present_Yes`, etc.
  - Razon: evita imponer orden artificial en categorias sin jerarquia.

### C) Estandarizacion
- Variables numericas se escalan con z-score.
- Razon: beneficia algoritmos basados en gradiente/distancia (incluidas redes neuronales).


## 6) Variantes rapidas por tipo de ejercicio

- **Clasificacion**:
  - `target_col = 'Outcome'` (binaria) o `target_col = 'Diagnosis'` (multiclase).

- **Regresion**:
  - Definir, por ejemplo, `target_col = 'CRP_Level'` y ajustar pipeline para no codificar `y`.

- **Red neuronal**:
  - Reutilizar `X_processed`, `y_train`, `y_test` y alimentar a Keras/PyTorch.

Este cuaderno deja el flujo de preprocesamiento listo para cualquiera de esas tres rutas.
